In [ ]:
import pandas as pd
import re
from google.colab import files

# Upload da planilha
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

# Use read_excel se for .xlsx, ou read_csv se for .csv
df = pd.read_excel(nome_arquivo)
# df = pd.read_csv(nome_arquivo, sep=';')  # alternativa se for csv

# Converte PRODUCAO para float (troca vírgula por ponto)
df['PRODUCAO'] = df['PRODUCAO'].astype(str).str.replace(',', '.').astype(float)

# Regex que captura: código - nome do produto - (dose)
padrao = re.compile(r'(\d+)\s*-\s*(.+?)\s*-\s*\((\d+,\d+)\)')

registros = []
for _, row in df.iterrows():
    producao = row['PRODUCAO']
    for codigo, nome, dose_str in padrao.findall(str(row['INSUMO_DOSE'])):
        dose = float(dose_str.replace(',', '.'))
        registros.append({
            'DATA': row['DATA'],
            'CODIGO': codigo,
            'PRODUTO': nome.strip(),
            'DOSE_HA': dose,
            'PRODUCAO': producao,
            'TOTAL_USADO': dose * producao
        })

df_detalhe = pd.DataFrame(registros)

# Resumo total por produto
resumo = (df_detalhe
          .groupby(['CODIGO', 'PRODUTO'])['TOTAL_USADO']
          .sum()
          .reset_index()
          .sort_values('TOTAL_USADO', ascending=False))

print(resumo)

In [ ]:
import pandas as pd
from google.colab import files

# Upload da planilha
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

df = pd.read_excel(nome_arquivo)
# df = pd.read_csv(nome_arquivo, sep=';')  # se for csv

# Garante que a coluna de área está como número
df['Area apontada (ha)'] = (
    df['Area apontada (ha)'].astype(str).str.replace(',', '.').astype(float)
)

# Junta pares Insumo/Dose de 1 a 5 em formato longo
linhas = []
for _, row in df.iterrows():
    for i in range(1, 6):  # Insumo 1..5 / Dose 1..5
        insumo = row.get(f'Insumo {i}')
        dose = row.get(f'Dose {i}')
        if pd.notna(insumo) and str(insumo).strip() != '' and pd.notna(dose):
            dose_val = float(str(dose).replace(',', '.'))
            linhas.append({
                'Operacao': row['Operacao'],
                'Data': row['Data'],
                'Gleba': row['Gleba'],
                'Quadra': row['Quadra'],
                'Area': row['Area apontada (ha)'],
                'Insumo': str(insumo).strip(),
                'Dose': dose_val,
                'Total_Usado': dose_val * row['Area apontada (ha)']
            })

df_long = pd.DataFrame(linhas)

# Total usado por produto (somando todas as ocorrências)
resumo = (df_long.groupby('Insumo')['Total_Usado']
          .sum()
          .reset_index()
          .sort_values('Total_Usado', ascending=False))

resumo.columns = ['Insumo', 'Total_Usado']
print(resumo)

# Exporta os dois: detalhado (df_long) e resumo (total por produto)
with pd.ExcelWriter('resultado_insumos.xlsx') as writer:
    df_long.to_excel(writer, sheet_name='Detalhado', index=False)
    resumo.to_excel(writer, sheet_name='Total_por_Produto', index=False)

files.download('resultado_insumos.xlsx')

In [ ]:
# Filtra só as linhas de glifosato no df_long (detalhado)
df_glifosato = df_long[df_long['Insumo'].str.upper().str.strip() == 'GLIFOSATO']

print(df_glifosato[['Operacao', 'Data', 'Gleba', 'Quadra', 'Area', 'Dose', 'Total_Usado']])
print('Soma:', df_glifosato['Total_Usado'].sum())
print('Número de linhas:', len(df_glifosato))

In [ ]:
# Dose média SIMPLES (o que provavelmente você calculou)
print('Média simples:', df_glifosato['Dose'].mean())

# Dose média PONDERADA pela área (a que reflete o total real)
media_ponderada = (df_glifosato['Dose'] * df_glifosato['Area']).sum() / df_glifosato['Area'].sum()
print('Média ponderada pela área:', media_ponderada)

# Confere: média ponderada x área total deve bater com o Total_Usado
print('Área total:', df_glifosato['Area'].sum())
print('Total via média ponderada:', media_ponderada * df_glifosato['Area'].sum())
print('Total via soma linha a linha:', df_glifosato['Total_Usado'].sum())